In [2]:
import pandas as pd
import numpy as np
import rasterio
import math
import os
from rasterio.transform import from_origin
from pathlib import Path
from tqdm import tqdm
from dask import delayed, compute
from natsort import natsorted

def tile_name_to_coord_bounds(tile_name):
    """Convert a tile name like 'N00E036' back to (min_lon, min_lat, max_lon, max_lat).
    
    Inverse of tile_name_from_latlon(). Tile names encode the SW corner;
    each tile spans 3° × 3°.
    """
    lat_prefix = tile_name[0]   # 'N' or 'S'
    lat_val = int(tile_name[1:3])  # e.g. '06' → 6
    lon_prefix = tile_name[3] # 'E' or 'W'
    lon_val = int(tile_name[4:7])  # e.g. '033' → 33

    min_lat = lat_val if lat_prefix == 'N' else -lat_val
    min_lon = lon_val if lon_prefix == 'E' else -lon_val

    max_lat = min_lat + 3
    max_lon = min_lon + 3

    return (min_lon, min_lat, max_lon, max_lat)


def df_to_geotiff(df, output_file, raster_bounds, buffer_size = 4, resolution = 0.00018, update_bar = None):
    """
    Convert a DataFrame of lat/lon points into a GeoTiff with dilation
    
    Args:
        df: DataFrame with columns ['lat', 'lon']
        output_file: Path to the output GeoTiff file
        raster_bounds: Tuple of (min_lon, min_lat, max_lon, max_lat) defining the raster extent
        buffer_size: Number of pixels to dilate around each detection (4 ~ 80m, 12 ~ 240m)
        resolution: Pixel size in degrees (0.00018 ~ 20m at equator)
    """
    if df.empty:
        raise ValueError("DataFrame is empty. Nothing to rasterize.")

    # Extract lat/lon coordinates for the flood detections
    lats = df['lat'].values
    lons = df['lon'].values

    # Raster bounds
    min_lon, min_lat, max_lon, max_lat = raster_bounds

    # Calculate raster dimensions
    width = int(math.ceil((max_lon - min_lon) / resolution)) + 1
    height = int(math.ceil((max_lat - min_lat) / resolution)) + 1

    # Transform for the raster
    transform = from_origin(min_lon, max_lat, resolution, resolution)

    # Initialize an empty raster
    raster = np.zeros((height, width), dtype = np.uint8)

    # Burn points with dilation
    for lon, lat in zip(lons, lats):
        row = int((max_lat - lat) / resolution)
        col = int((lon - min_lon) / resolution)

        r_start, r_end = max(0, row - buffer_size), min(height, row + buffer_size + 1)
        c_start, c_end = max(0, col - buffer_size), min(width, col + buffer_size + 1)

        raster[r_start:r_end, c_start:c_end] = 1

    # Write GeoTIFF
    meta = {
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'count': 1,
        'dtype': 'uint8',
        'crs': 'EPSG:4326',
        'transform': transform,
        'compress': 'lzw'
    }

    with rasterio.open(output_file, 'w', **meta) as dst:
        dst.write(raster, 1)

    if update_bar:
        update_bar()

    return True






In [ ]:
# Define root directory for flood data
root_dir = Path("/media/eric/eric_backup_01/flood_data")

# Search for -post-processing.parquet
parquet_files = natsorted(list(root_dir.rglob("*-post-processing.parquet")))

# Load parquet data
with tqdm(total=len(parquet_files), desc="\nProcessing Parquet Files") as pbar:
    for parquet_file in parquet_files:
        df = pd.read_parquet(parquet_file)
        print(f"\nLoaded {parquet_file.name}")
        # base_path = Path("/media/eric/eric_backup_01/flood_data/S06/S06E039")
        # df = pd.read_parquet(base_path / "S06E039-post-processing.parquet")

        # Create a 'date' column from 'year', 'month', 'day'
        df['date'] = pd.to_datetime(df[['year', 'month', 'day']])

        # # Define parquet filters
        filter_params = {
            "dem_metric_2_max": 10,
            "soil_moisture_sca_min": 1,
            "soil_moisture_zscore_min": 1,
            "soil_moisture_min": 20,
            "temp_min": 0,
            "exclude_land_cover": 60,
            "edge_fp_eq": 0
        }

        mask = (
            (df.dem_metric_2 < filter_params["dem_metric_2_max"]) &
            (df.soil_moisture_sca > filter_params["soil_moisture_sca_min"]) &
            (df.soil_moisture_zscore > filter_params["soil_moisture_zscore_min"]) &
            (df.soil_moisture > filter_params["soil_moisture_min"]) &
            (df.temp > filter_params["temp_min"]) &
            (df.land_cover != filter_params["exclude_land_cover"]) &
            (df.edge_false_positives == filter_params["edge_fp_eq"])
            )

        # Apply the mask to filter the DataFrame
        df = df[mask]



        # Load dates for when floods were detected in tile S06E039
        dates = pd.read_csv(parquet_file.parent / f"{parquet_file.parent.name}-post-processing_flood_dates.csv")
        dates['date'] = pd.to_datetime(dates['date'])

        assert dates['date'].unique().shape[0] == df['date'].unique().shape[0], "Mismatch in unique dates between flood dates and filtered data"
        
        # Create bounds for region of interest based on tile name
        raster_bounds = tile_name_to_coord_bounds(parquet_file.parent.name)
        print(f"Raster bounds for tile {parquet_file.parent.name}: {raster_bounds}")   

        
        # Create 240m buffered GeoTIFFs for each date in the filtered DataFrame
        target_buffer_size = 12  # 12 pixels ~ 240m at 0.00018 resolution
        target_path = parquet_file.parent / f"FLOOD_TARGETS_{target_buffer_size*20}m"
        target_path.mkdir(parents=True, exist_ok=True)
        assert target_path.exists(), f"Failed to create target path: {target_path}"

        with tqdm(total=len(dates), desc="Creating Flood Targets") as bar:
            delayed_tasks = []
            for i, date in enumerate(dates['date']):
                date_str = date.strftime('%Y_%m_%d')
                output_file = target_path / f"{parquet_file.parent.name}_flood_targets_{date_str}.tif"

                # Filter the DataFrame for the current date
                df_date = df[df['date'] == date]
                
                # Create GeoTIFF for the current date
                delayed_tasks.append(delayed(df_to_geotiff)(df_date, output_file, raster_bounds, buffer_size=target_buffer_size, resolution=0.00018, update_bar=bar.update))

            # Execute all delayed tasks in parallel
            compute(*delayed_tasks, scheduler = "threads", num_workers = min(16, os.cpu_count()//2))

    pbar.update(1)


Processing Parquet Files:   0%|          | 0/1 [00:00<?, ?it/s]


Loaded S06E039-post-processing.parquet
Raster bounds for tile S06E039: (39, -6, 42, -3)


Creating Flood Targets:   2%|▏         | 6/340 [00:01<00:58,  5.67it/s]

Processing Parquet Files: 100%|██████████| 1/1 [00:01<00:00,  1.50s/it]
